# Optimizing QSD circuits
Assume no backend

In [1]:
import sys
sys.path.append("../")
from utils.utils import *

In [5]:
!ls

2024-06-23			     Phi_q5_n3_symm.npy
20250912_benchmark_gen.log	     Phi_q6_n3_s42.npy
20250913.log			     Phi_tilde_0731_test.npy
20250922.log			     Phi_tilde_0809_test.npy
capture_list.txt		     Phi_tilde_0813_test.npy
circuits			     Phi_tilde_0902_test.npy
coh5.log			     Phi_tilde_q1_n2_s42.npy
coh.log				     Phi_tilde_q2_n2_s42.npy
convergence_plot.png		     Phi_tilde_q2_n3_s42.npy
dev.log				     Phi_tilde_q3_n3_s42.npy
exp0				     Phi_tilde_q3_n3_symm.npy
experiment_new.py		     Phi_tilde_q4_n3_s42.npy
experiments.py			     Phi_tilde_q5_n3_s42.npy
exp_uqsd_med.py			     Phi_tilde_q5_n3_symm.npy
figures				     Phi_tilde_q6_n3_s42.npy
flow				     plot_results.py
frio.csv			     process_results.py
initial_configuration.png	     psucc_results_3states.csv
iso_coh_q2_n3.npy		     psucc_results_a1_0_001_2states.csv
iso_coh_q3_n3_med_fullpovm_csd.npy   psucc_results_a1_0_001_3states.csv
iso_coh_q3_n3.npy		     psucc_results.csv
iso_coh_q5_n3.npy		     pyproject.toml
iso_med_coh_q3

In [2]:
from qiskit.circuit import QuantumCircuit
from qiskit.quantum_info import Operator
from qiskit import transpile


def get_unitary_circuit(qc) -> QuantumCircuit:
    # https://docs.quantum.ibm.com/guides/synthesize-unitary-operators#synthesize-unitary-operations

    U = Operator(qc)

    nq = qc.num_qubits
    unitary_circuit = QuantumCircuit(nq)
    unitary_circuit.unitary(U, range(nq))
    return unitary_circuit


def resynth_unitary(qc) -> QuantumCircuit:
    return get_unitary_circuit(qc).decompose(reps=3)


def resynth_unitary_approx(qc) -> QuantumCircuit:
    tmp_circuit = get_unitary_circuit(qc)

    # The approximation degree defaults to 1.0
    approx_circuit = transpile(
        tmp_circuit,
        unitary_synthesis_method="aqc",
        unitary_synthesis_plugin_config={"seed": 42},
        approximation_degree=1.0,
        seed_transpiler=11,
    )

    return approx_circuit.decompose(reps=3)


def resynth_unitary_approx_max(qc) -> QuantumCircuit:
    tmp_circuit = get_unitary_circuit(qc)

    # It won't work without specifying aqc
    # It won't work without specifying basis_gates?
    approx_circuit = transpile(
        tmp_circuit,
        unitary_synthesis_method="aqc",
        unitary_synthesis_plugin_config={"seed": 42},
        approximation_degree=0,
        basis_gates=['u', 'cx'],
        seed_transpiler=11,
    )

    return approx_circuit.decompose(reps=3)


def resynth_aqc(qc, uni_synth_config=None) -> QuantumCircuit:
    tmp_circuit = get_unitary_circuit(qc)

    aqc_circuit = transpile(
        tmp_circuit,
        unitary_synthesis_method="aqc",
        unitary_synthesis_plugin_config=uni_synth_config,
        approximation_degree=0,
        seed_transpiler=11,
    )

    return aqc_circuit.decompose(reps=3)

In [3]:
def try_synth(qc, case_id):
    # Show different results (depth and fidelity)
    resynth_circuit = resynth_unitary(qc)
    approx_circuit = resynth_unitary_approx(qc)
    approx_max_circuit = resynth_unitary_approx_max(qc)
    aqc_circuit_v0 = resynth_aqc(
        qc,
        {
            # "network_layout": "cart",
            "connectivity_type": "star",
            "depth": 10,
            "seed": 3,
        },
    )
    # TODO Approx again
    # Statistics

    print(qc.count_ops())
    print(qc.depth())
    print(resynth_circuit.count_ops())
    print(resynth_circuit.depth())
    print(approx_circuit.count_ops())
    print(approx_circuit.depth())
    print(approx_max_circuit.count_ops())
    print(approx_max_circuit.depth())
    print(aqc_circuit_v0.count_ops())
    print(aqc_circuit_v0.depth())

    from qiskit.quantum_info import process_fidelity

    # Two operators which differ only by phase
    op_a = Operator(qc)
    op_b = Operator(resynth_circuit)
    op_c = Operator(approx_circuit)
    op_c_1 = Operator(approx_max_circuit)
    op_d = Operator(aqc_circuit_v0)

    # Compute process fidelity
    F_resynth = process_fidelity(op_a, op_b)
    print("Process fidelity (resynth) =", F_resynth)
    F_approx = process_fidelity(op_a, op_c)
    print("Process fidelity (approx) =", F_approx)
    F_approx_max = process_fidelity(op_a, op_c_1)
    print("Process fidelity (approx_max) =", F_approx_max)
    F_aqc = process_fidelity(op_a, op_d)
    print("Process fidelity (aqc) =", F_aqc)

    import qiskit.qasm2

    qiskit.qasm2.dump(
        resynth_circuit,
        f"qc_iso_{case_id}_no_backend_resynth.qasm",
    )
    qiskit.qasm2.dump(
        approx_circuit,
        f"qc_iso_{case_id}_no_backend_approx.qasm",
    )
    qiskit.qasm2.dump(
        approx_circuit,
        f"qc_iso_{case_id}_no_backend_approx_max.qasm",
    )
    qiskit.qasm2.dump(
        aqc_circuit_v0,
        f"qc_iso_{case_id}_no_backend_aqc.qasm",
    )
    return

In [4]:
try_synth(
    QuantumCircuit.from_qasm_file(
        "circuits/coherent/symm/coh_symm_q4_n3_optuqsd_reducedpovm_no_backend.qasm"
    ),
    "coh_q4_n3",
)

FileNotFoundError: /home/ChienKaiMa/QSD/circuits/coherent/symm/coh_symm_q4_n3_optuqsd_reducedpovm_no_backend.qasm

In [ ]:
try_synth(
    QuantumCircuit.from_qasm_file(
        get_qasm_name_by_case(
            get_case_id(3, 7, 3),
        ),
    ),
    get_case_id(3, 7, 3),
)

OrderedDict({'u': 308, 'cx': 241, 'rz': 15})
470
OrderedDict({'u': 208, 'cx': 100})
225
OrderedDict({'u': 126, 'cx': 61})
83
OrderedDict({'u': 126, 'cx': 61})
83
OrderedDict({'u': 24, 'cx': 10})
15
Process fidelity (resynth) = 0.9999999999999987
Process fidelity (approx) = 0.9999165722755792
Process fidelity (approx_max) = 0.9999165722755794
Process fidelity (aqc) = 0.4604218903740472


In [ ]:
# Different configs
# aqc_config
{
    "network_layout": {"sequ", "spin", "cart", "cyclic_spin", "cyclic_line"},
    "connectivity_type": {"full", "line", "star"},
    "depth": 30,
    # "optimizer": ,
    "seed": 42,
    # "initial_point",
}

{'network_layout': {'cart', 'cyclic_line', 'cyclic_spin', 'sequ', 'spin'},
 'connectivity_type': {'full', 'line', 'star'},
 'depth': 30,
 'seed': 42}

In [ ]:
from qiskit.quantum_info import process_fidelity

qc = QuantumCircuit.from_qasm_file(
    get_qasm_name_by_case(
        get_case_id(3, 7, 3),
    ),
)

op_a = Operator(qc)

print(qc.count_ops())
print(qc.depth())

for i in range(5):
    opt_qc = resynth_aqc(
        qc,
        {
            "network_layout": "spin",
            "connectivity_type": "star",
            "depth": 10 + 10 * i,
            "seed": 42,
        },
    )
    print(opt_qc.count_ops())
    print(opt_qc.depth())

    op_b = Operator(opt_qc)

    # Compute process fidelity
    F_resynth = process_fidelity(op_a, op_b)
    print("Process fidelity (resynth) =", F_resynth)
    print()

OrderedDict({'u': 308, 'cx': 241, 'rz': 15})
470
OrderedDict({'u': 24, 'cx': 10})
15
Process fidelity (resynth) = 0.44638274440967257

OrderedDict({'u': 44, 'cx': 20})
27
Process fidelity (resynth) = 0.6716499586466799

OrderedDict({'u': 64, 'cx': 30})
41
Process fidelity (resynth) = 0.852718145415536

OrderedDict({'u': 84, 'cx': 40})
55
Process fidelity (resynth) = 0.948766607235006

OrderedDict({'u': 104, 'cx': 50})
67
Process fidelity (resynth) = 0.9907631900728991



In [ ]:
from qiskit.quantum_info import process_fidelity

qc = QuantumCircuit.from_qasm_file(
    get_qasm_name_by_case(
        get_case_id(3, 7, 3),
    ),
)

op_a = Operator(qc)

print(qc.count_ops())
print(qc.depth())

for i in range(5):
    opt_qc = resynth_aqc(
        qc,
        {
            "network_layout": "cart",
            "connectivity_type": "star",
            "depth": 10 + 10 * i,
            "seed": 42,
        },
    )
    print(opt_qc.count_ops())
    print(opt_qc.depth())

    op_b = Operator(opt_qc)

    # Compute process fidelity
    F_resynth = process_fidelity(op_a, op_b)
    print("Process fidelity (resynth) =", F_resynth)
    print()

OrderedDict({'u': 308, 'cx': 241, 'rz': 15})
470
OrderedDict({'u': 244, 'cx': 120})
211
Process fidelity (resynth) = 0.9999999985851282

OrderedDict({'u': 244, 'cx': 120})
211
Process fidelity (resynth) = 0.9999999985851282

OrderedDict({'u': 244, 'cx': 120})
211
Process fidelity (resynth) = 0.9999999985851282

OrderedDict({'u': 244, 'cx': 120})
211
Process fidelity (resynth) = 0.9999999985851282

OrderedDict({'u': 244, 'cx': 120})
211
Process fidelity (resynth) = 0.9999999985851282



In [ ]:
from qiskit.quantum_info import process_fidelity

qc = QuantumCircuit.from_qasm_file(
    get_qasm_name_by_case(
        get_case_id(3, 7, 3),
    ),
)

op_a = Operator(qc)

print(qc.count_ops())
print(qc.depth())

for i in range(5):
    opt_qc = resynth_aqc(
        qc,
        {
            "network_layout": "sequ",
            "connectivity_type": "star",
            "depth": 10 + 10 * i,
            "seed": 42,
        },
    )
    print(opt_qc.count_ops())
    print(opt_qc.depth())

    op_b = Operator(opt_qc)

    # Compute process fidelity
    F_resynth = process_fidelity(op_a, op_b)
    print("Process fidelity (resynth) =", F_resynth)
    print()

OrderedDict({'u': 308, 'cx': 241, 'rz': 15})
470
OrderedDict({'u': 24, 'cx': 10})
21
Process fidelity (resynth) = 0.4922817677967132

OrderedDict({'u': 44, 'cx': 20})
41
Process fidelity (resynth) = 0.6771025915257053

OrderedDict({'u': 64, 'cx': 30})
61
Process fidelity (resynth) = 0.842026314945232

OrderedDict({'u': 84, 'cx': 40})
81
Process fidelity (resynth) = 0.9543852337126041

OrderedDict({'u': 104, 'cx': 50})
101
Process fidelity (resynth) = 0.9936371102023243



In [ ]:
from qiskit.quantum_info import process_fidelity

qc = QuantumCircuit.from_qasm_file(
    get_qasm_name_by_case(
        get_case_id(3, 7, 3),
    ),
)

op_a = Operator(qc)

print(qc.count_ops())
print(qc.depth())

for i in range(5):
    opt_qc = resynth_aqc(
        qc,
        {
            "network_layout": "cyclic_line",
            "connectivity_type": "line",
            "depth": 10 + 10 * i,
            "seed": 42,
        },
    )
    print(opt_qc.count_ops())
    print(opt_qc.depth())

    op_b = Operator(opt_qc)

    # Compute process fidelity
    F_resynth = process_fidelity(op_a, op_b)
    print("Process fidelity (resynth) =", F_resynth)
    print()

OrderedDict({'u': 308, 'cx': 241, 'rz': 15})
470
OrderedDict({'u': 24, 'cx': 10})
21
Process fidelity (resynth) = 0.3780482806194589

OrderedDict({'u': 44, 'cx': 20})
41
Process fidelity (resynth) = 0.6190447810449327

OrderedDict({'u': 64, 'cx': 30})
61
Process fidelity (resynth) = 0.8288406644296822

OrderedDict({'u': 84, 'cx': 40})
81
Process fidelity (resynth) = 0.9377110542941772

OrderedDict({'u': 104, 'cx': 50})
101
Process fidelity (resynth) = 0.990647056622718

